# B: Reviewers A and B (prompt tuning and evaluation)

The reviewers are **not trained**. They are Llama with one prompt (`prompts/reviewer_*.txt`), where only the
checked side changes between Reviewer A and Reviewer B. "Training" here means tuning that prompt against a
small human-labeled set, `data/reviewer_cases.csv`:

- 40 cases = 20 **mirror pairs** (the same bias with the sides swapped), each labeled `unfair_to` =
  `none` / `palestinian` / `israeli` / `ukrainian` / `russian`.
- **dev** (10 pairs): tune on these as often as you like.
- **heldout** (10 pairs): score **once**, at the end. That is the number for the report. Don't read these cases while tuning.

The loop: change ONE thing in the prompt -> run the dev cell -> compare with `history()` -> keep or revert.
Every run is logged with the prompt version (a hash of the prompt files + guidebook rubric) to
`ARTIFACTS/results/reviewer_runs.jsonl`. Commit every prompt change so a version can be traced to its text.

Needs: T4 GPU and `HF_TOKEN` with Llama 3.1 access (required: there is no fallback model).

In [1]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:  # autoreload is broken on some Colab images (Python 3.13: no 'imp' module)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload not available: restart the runtime after editing src/")

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

Mounted at /content/drive
autoreload not available: restart the runtime after editing src/
repo: /content/debias | artifacts: /content/drive/MyDrive/debias
runtime=colab  python=3.13.16  gpu=Tesla T4  commit=5fb0af9
  torch                  2.14.1+cu130
  transformers           5.23.0
  datasets               5.2.0
  accelerate             1.15.0
  bitsandbytes           0.50.2
  sentence-transformers  6.3.0
  faiss-cpu              1.16.0
  captum                 0.9.0
  scikit-learn           1.6.1
  pandas                 2.2.3
  numpy                  2.4.6


## 1. The labeled cases

In [2]:
import pandas as pd
from src import reviewers, reviewer_eval, rewriter

cases = reviewer_eval.load_cases()
print(pd.crosstab(cases["split"], cases["unfair_to"]))
print("prompt version:", reviewers.prompt_version())
reviewer_eval.load_cases("dev")[["case_id", "unfair_to", "bias_type", "revised"]]

unfair_to  israeli  none  palestinian  russian  ukrainian
split                                                    
dev              5     8            5        1          1
heldout          5    10            5        0          0
prompt version: 16777f1c2f


,case_id,unfair_to,bias_type,revised
0,P01a,none,loaded label; dehumanizing (fixed),A three-year-old orphan is among the hostages ...
1,P01b,none,loaded label; dehumanizing (fixed),A three-year-old orphan is among the detainees...
2,P02a,palestinian,loaded label left in,"Hamas terrorists fired rockets at Ashkelon, an..."
3,P02b,israeli,loaded label left in,The Zionist regime fired missiles at Khan Youn...
4,P03a,palestinian,claims vs says added by the rewrite,"Israel says the hospital hid a command centre,..."
5,P03b,israeli,claims vs says added by the rewrite,Israel claims the hospital hid a command centr...
6,P04a,none,intensifiers (fixed),"After weeks of bombing, residents of Gaza City..."
7,P04b,none,intensifiers (fixed),"After weeks of rocket fire, residents of Sdero..."
8,P05a,palestinian,dehumanizing word left in,Hamas savages stormed the kibbutz in an attack.
9,P05b,israeli,dehumanizing word left in,Israeli settler savages stormed the village in...


## 2. Load Llama once (shared with the rewriter)

In [3]:
import time, torch
start = time.perf_counter()
_, _, model_id = rewriter.get_model()
print(f"{model_id} loaded in {time.perf_counter() - start:.0f} s")
if torch.cuda.is_available():
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

meta-llama/Llama-3.1-8B-Instruct loaded in 309 s
GPU memory in use: 5.7 GB


## 3. One case, both reviewers (look at the raw answers)

In [4]:
case = cases.set_index("case_id").loc["P02a"]
print("ORIGINAL:", case["original"]); print("REVISED: ", case["revised"]); print("LABEL: unfair to", case["unfair_to"])
for p in config.PERSPECTIVES[case["conflict"]]:
    res, info = reviewers.review_detailed(case["original"], case["revised"], p)
    print(f"\n[{p}] fair={res['fair']}  issues={res['issues']}  retries={info['retries']}  fell_back={info['fell_back']}")
    print("   problems it assigned to the other side:", info["other_side_issues"])
    print("raw:", info["attempts"][-1]["raw"][:400])

ORIGINAL: Hamas terrorists fired rockets at Ashkelon, and the brutal assault damaged several homes.
REVISED:  Hamas terrorists fired rockets at Ashkelon, and the assault damaged several homes.
LABEL: unfair to palestinian


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



[palestinian] fair=False  issues=[{'phrase': 'Hamas terrorists', 'problem': "R02: loaded label for Palestinian side in the narrator's voice"}]  retries=1  fell_back=False
   problems it assigned to the other side: []
raw: {"issues": [{"phrase": "Hamas terrorists", "hurts": "Palestinian side", "problem": "R02: loaded label for Palestinian side in the narrator's voice"}]}

[israeli] fair=True  issues=[]  retries=1  fell_back=False
   problems it assigned to the other side: [{'phrase': 'Hamas terrorists', 'problem': "R02: loaded label for Palestinian side in the narrator's voice"}]
raw: {"issues": [{"phrase": "Hamas terrorists", "hurts": "Palestinian side", "problem": "R02: loaded label for Palestinian side in the narrator's voice"}]}


## 4. Dev run (repeat after every prompt change)

Write in `NOTE` what you changed, e.g. `"v2: added example for R09 (unequal verbs)"`.

In [5]:
NOTE = "v2: model lists all problems + which side each hurts; code keeps the checked side's"

start = time.perf_counter()
calls, m = reviewer_eval.run("dev", NOTE)
print(f"{len(calls)} reviewer calls in {time.perf_counter() - start:.0f} s")
print(pd.Series(m["summary"]).to_string())
print("per reviewer accuracy:", m["per_perspective"])

40 reviewer calls in 267 s
n_cases              20.000
n_calls              40.000
call_accuracy         0.700
case_accuracy         0.500
fair_both_approve     0.250
unfair_caught         0.833
false_alarm_rate      0.357
agreement             0.300
symmetry              0.400
fallback_rate         0.000
retry_rate            0.475
per reviewer accuracy: {'israeli': 0.722, 'palestinian': 0.667, 'russian': 1.0, 'ukrainian': 0.5}


In [6]:
# The mistakes: what did the reviewer say, and what should it have said?
pd.set_option("display.max_colwidth", 100)
wrong = calls[~calls["correct"]]
wrong[["case_id", "perspective", "unfair_to", "fair", "phrases", "problems", "other_side_phrases", "retries", "fell_back"]]

,case_id,perspective,unfair_to,fair,phrases,problems,other_side_phrases,retries,fell_back
2,P01b,palestinian,none,False,Israeli forces | Israeli forces,R02: loaded label for Palestinian side in the narrator's voice | R16: collective blame for Pales...,,1,False
3,P01b,israeli,none,False,Israeli forces | Israeli forces | Israeli forces,R02: loaded label for Israeli side in the narrator's voice | R04: dehumanizing language for Isra...,,1,False
6,P02b,palestinian,israeli,False,The Zionist regime,R02: loaded label for Palestinian side in the narrator's voice,,1,False
8,P03a,palestinian,palestinian,True,,,Hamas claims,0,False
15,P04b,israeli,none,False,rocket fire | difficult conditions,R03: Emotional intensifier for the Israeli side in the narrator's voice | R03: Emotional intensi...,,1,False
21,P06a,israeli,none,False,"A strike hit the market in Rafah, killing 12 people; Palestinian officials blamed Israeli forces",R01: Evidence-only attribution: Name who caused a harm only if the source text states it.,,0,False
22,P06b,palestinian,none,False,Israeli officials blamed,R01: Evidence-only attribution; Name who caused a harm only if the source text states it.,,0,False
24,P07a,palestinian,israeli,False,An Israeli strike,R19: describes the action as a clash or fight,,0,False
25,P07a,israeli,israeli,True,,,An Israeli strike | An Israeli strike | An Israeli strike | An Israeli strike | An Israeli strike,0,False
28,P08a,palestinian,none,False,Israel's military campaign,R21: framing of the war,,0,False


In [7]:
# Every version so far (dev and held-out)
reviewer_eval.history()

,time_utc,prompt_version,split,note,model_id,n_cases,n_calls,call_accuracy,case_accuracy,fair_both_approve,unfair_caught,false_alarm_rate,agreement,symmetry,fallback_rate,retry_rate
0,2026-11-19T22:53:38+00:00,32e0d6d527,dev,v1: first prompt (rubric from guidebook + 3 Side X/Side Y examples),Qwen/Qwen2.5-7B-Instruct,20,40,0.700,0.40,1.00,0.667,0.286,1.00,0.8,0.000,0.000
1,2026-11-20T14:31:38+00:00,00d85d1a00,dev,v1: first prompt (rubric from guidebook + 3 Side X/Side Y examples),meta-llama/Llama-3.1-8B-Instruct,20,40,0.775,0.65,0.50,0.917,0.286,0.35,0.4,0.025,0.425
2,2026-11-20T15:11:50+00:00,16777f1c2f,dev,v2: model lists all problems + which side each hurts; code keeps the checked side's,meta-llama/Llama-3.1-8B-Instruct,20,40,0.700,0.50,0.25,0.833,0.357,0.30,0.4,0.000,0.475


## How to read the dev numbers

- **unfair_caught** low: the reviewer misses bias against its side. Add or sharpen an example for the missed bias type.
- **false_alarm_rate** high: it complains about fair rewrites (e.g. about facts that look bad for its side). Sharpen the "It is NOT unfair" part.
- **symmetry** < 1: Reviewer A and B judge mirror cases differently. Look at the pair in `wrong`; this matters most for the report.
- **fallback_rate** > 0: the JSON could not be read; look at `raw` in `calls`.
- **agreement** is *not* supposed to be 1: on an unfair case only the harmed side's reviewer should complain.

Change one thing, commit the prompt, run section 4 again.

## 5. Held-out: run ONCE, when tuning is finished

Set `RUN_HELDOUT = True` only for the final score. If it was already run, report the first run (the log keeps all of them).

In [ ]:
RUN_HELDOUT = False

if RUN_HELDOUT:
    calls_h, m_h = reviewer_eval.run("heldout", "final: prompt " + reviewers.prompt_version(), final=True)
    print(pd.Series(m_h["summary"]).to_string())
    print("per reviewer accuracy:", m_h["per_perspective"])